In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

IMG_SIZE = 160
BATCH_SIZE = 16

train_dir = "/kaggle/input/teeth-ds/teeth_data_set/Training"
val_dir = "/kaggle/input/teeth-ds/teeth_data_set/Validation"

# =========================
# Data Augmentation (صح)
# =========================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # بدل rescale

    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.15,
    brightness_range=[0.8,1.2],
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input  # بدل rescale
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes

Found 3702 images belonging to 7 classes.
Found 1058 images belonging to 7 classes.


In [6]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, BatchNormalization
from tensorflow.keras.models import Sequential

# =========================
# Build CNN
# =========================
model = Sequential()

# Block 1
model.add(Conv2D(32, (3,3), activation="relu", padding="same",
                 input_shape=(IMG_SIZE, IMG_SIZE, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Block 2
model.add(Conv2D(64, (3,3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Block 3
model.add(Conv2D(128, (3,3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Block 4 (زيادة قوة الموديل)
model.add(Conv2D(256, (3,3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Flatten
model.add(Flatten())

# Dense layers
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))

# Output
model.add(Dense(NUM_CLASSES, activation="softmax"))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 160, 160, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 80, 80, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 40, 40, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 40, 40, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 20, 20, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 20, 20, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 10, 10, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │     6,553,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,977,991 (26.62 MB)

 Trainable params: 6,977,031 (26.62 MB)

 Non-trainable params: 960 (3.75 KB)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# =========================
# Callbacks (مظبوطة)
# =========================
callbacks = [

    # يوقف بدري قبل ما يبوّظ
    EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    # يقلل LR لما يقف
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    # يحفظ أفضل موديل
    ModelCheckpoint(
        "best_teeth_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

In [9]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [11]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=60,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/60
232/232 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.2149 - loss: 2.6297
Epoch 1: val_accuracy improved from -inf to 0.33365, saving model to best_teeth_model.keras
232/232 ━━━━━━━━━━━━━━━━━━━━ 43s 157ms/step - accuracy: 0.2150 - loss: 2.6279 - val_accuracy: 0.3336 - val_loss: 1.7468 - learning_rate: 1.0000e-04
Epoch 2/60
232/232 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.2627 - loss: 1.8441
Epoch 2: val_accuracy improved from 0.33365 to 0.36389, saving model to best_teeth_model.keras
232/232 ━━━━━━━━━━━━━━━━━━━━ 30s 128ms/step - accuracy: 0.2627 - loss: 1.8441 - val_accuracy: 0.3639 - val_loss: 1.6488 - learning_rate: 1.0000e-04
Epoch 3/60
232/232 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.2876 - loss: 1.7691
Epoch 3: val_accuracy improved from 0.36389 to 0.40265, saving model to best_teeth_model.keras
232/232 ━━━━━━━━━━━━━━━━━━━━ 30s 128ms/step - accuracy: 0.2876 - loss: 1.7690 - val_accuracy: 0.4026 - val_loss: 1.5828 - learning_rate: 1.0000e-04
Epoch 4/60

In [12]:
from tensorflow.keras.models import load_model

best_model = load_model("best_teeth_model.keras")

In [14]:
test_dir = "/kaggle/input/teeth-ds/teeth_data_set/Testing"

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(160,160),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

Found 533 images belonging to 7 classes.


In [15]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

preds = best_model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)

print(classification_report(test_gen.classes, y_pred))

34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        80
           1       0.95      0.96      0.95        76
           2       0.97      0.86      0.91        71
           3       0.84      0.89      0.86        80
           4       0.91      0.89      0.90        70
           5       0.87      0.81      0.84        80
           6       0.84      0.96      0.90        76

    accuracy                           0.90       533
   macro avg       0.90      0.90      0.90       533
weighted avg       0.90      0.90      0.90       533



In [16]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score

# ======================
# Load model
# ======================
model = load_model("best_teeth_model.keras")

# ======================
# Class names (بالترتيب)
# ======================
class_names = ['CaS','CoS','Gum','MC','OC','OLP','OT']

# ======================
# Test folder
# ======================
test_dir = "/kaggle/input/teeth-ds/teeth_data_set/Testing"

IMG_SIZE = 160

y_true = []
y_pred = []

# ======================
# Loop على الصور
# ======================
for class_idx, class_name in enumerate(class_names):

    class_path = os.path.join(test_dir, class_name)

    for img_name in os.listdir(class_path):

        img_path = os.path.join(class_path, img_name)

        # load image
        img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        # EfficientNet preprocessing
        img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

        # prediction
        pred = model.predict(img_array, verbose=0)
        pred_idx = np.argmax(pred)

        y_true.append(class_idx)
        y_pred.append(pred_idx)

        # طباعة actual vs predicted
        print(f"Actual: {class_name} | Predicted: {class_names[pred_idx]}")

# ======================
# Accuracy
# ======================
acc = accuracy_score(y_true, y_pred)

print("\n=====================")
print("Test Accuracy:", acc)
print("=====================")

Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: MC
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CoS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: OT
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Predicted: CaS
Actual: CaS | Pr